# rendering

> Queue panel and item rendering with callback-based custom content.

In [ ]:
#| default_exp rendering

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from typing import Any, Callable, List, Optional

from fasthtml.common import Div, Span, Button, Ul, Li, P, Hidden

from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_colors, badge_sizes
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui, border_dui

from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.utilities.typography import (
    font_size, font_weight, text_align, list_style
)
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.utilities.interactivity import cursor
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, justify, items, grow
)
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Design system recipes (V1 button roles, V10 panel variants, V11 icon-size roles)
from cjm_fasthtml_design_system.buttons import buttons
from cjm_fasthtml_design_system.panels import panels
from cjm_fasthtml_design_system.icons import icons

from cjm_fasthtml_sortable_queue.config import SortableQueueConfig, get_item_key
from cjm_fasthtml_sortable_queue.html_ids import SortableQueueHtmlIds
from cjm_fasthtml_sortable_queue.models import SortableQueueUrls

In [ ]:
#| export
def render_queue_item(
    config: SortableQueueConfig,  # Queue configuration
    ids: SortableQueueHtmlIds,  # HTML ID generators
    urls: SortableQueueUrls,  # URL endpoints
    item: dict,  # Queue item data
    index: int,  # 0-based position in queue
    render_content: Callable[[dict, int], Any],  # Callback for custom item content
    extra_attrs: Optional[dict] = None,  # Additional HTML attributes per item
) -> Any:  # Li element
    """Render a single queue item with drag handle, position, custom content, and remove button."""
    key = get_item_key(config, item)
    
    return Li(
        # Hidden input for Sortable.js form data collection
        Hidden(name="item", value=key),
        
        # Drag handle
        Span(
            lucide_icon("grip-vertical", size=icons.icon_button, cls=str(text_dui.base_content.opacity(40))),
            cls=combine_classes(config.handle_class, cursor.move, p.r(2))
        ),
        
        # Position number (1-based)
        Span(f"{index + 1}.", cls=combine_classes(font_weight.bold, p.r(2), w(6))),
        
        # Custom content from consumer callback
        render_content(item, index),
        
        # Remove button
        Button(
            lucide_icon("x", size=icons.icon_button, cls=str(text_dui.base_content.opacity(60))),
            cls=combine_classes(buttons.item_remove, m.l(1)),
            hx_post=urls.remove,
            hx_vals=json.dumps({"key": key}),
            hx_target=ids.as_selector(ids.container),
            hx_swap="outerHTML",
            data_action="remove",
            title="Remove from queue"
        ),
        
        id=ids.item(key),
        cls=combine_classes(
            config.item_class,
            flex_display,
            items.center,
            p(3),
            bg_dui.base_100,
            bg_dui.base_300.hover,
            overflow.x.auto
        ),
        data_key=key,
        **(extra_attrs or {})
    )

In [ ]:
#| export
def _render_default_empty() -> Any:  # Empty state element
    """Default empty state when no items are in the queue."""
    return Div(
        P(
            "No items selected",
            cls=combine_classes(text_dui.base_content.opacity(40), text_align.center, font_size.sm)
        ),
        P(
            "Add items from the browser",
            cls=combine_classes(text_dui.base_content.opacity(30), text_align.center, font_size.xs)
        ),
        cls=combine_classes(p(8), flex_display, flex_direction.col, justify.center, items.center, grow())
    )

In [ ]:
#| export
def render_sortable_queue(
    config: SortableQueueConfig,  # Queue configuration
    ids: SortableQueueHtmlIds,  # HTML ID generators
    urls: SortableQueueUrls,  # URL endpoints
    queue_items: List[dict],  # Ordered list of queue item dicts
    render_content: Callable[[dict, int], Any],  # Callback for custom item content
    render_empty: Optional[Callable[[], Any]] = None,  # Custom empty state (default provided)
    render_header_actions: Optional[Callable[[List[dict], SortableQueueUrls], Any]] = None,  # Custom header actions
    render_footer: Optional[Callable[[List[dict]], Any]] = None,  # Optional footer content
    extra_item_attrs: Optional[Callable[[dict], dict]] = None,  # Additional data-* attributes per item
    container_classes: tuple = (),  # Additional classes on container div
) -> Any:  # Queue panel element
    """Render the complete sortable queue panel."""
    count = len(queue_items)
    has_items = count > 0
    
    # Header actions: default is a Clear button when items exist
    if render_header_actions is not None:
        header_actions = render_header_actions(queue_items, urls)
    elif has_items:
        header_actions = Button(
            "Clear",
            cls=buttons.soft_utility,
            hx_post=urls.clear,
            hx_target=ids.as_selector(ids.container),
            hx_swap="outerHTML"
        )
    else:
        header_actions = None
    
    # Header
    header = Div(
        Div(
            Span(config.queue_title, cls=str(font_weight.bold)),
            Span(
                str(count),
                cls=combine_classes(badge, badge_colors.primary, badge_sizes.sm, m.l(2))
            ),
            cls=str(flex_display)
        ),
        header_actions,
        id=ids.header,
        cls=combine_classes(
            flex_display, justify.between, items.center,
            p(3), border_dui.base_200, border.b()
        )
    )
    
    # Body: queue list or empty state
    if has_items:
        li_items = [
            render_queue_item(
                config, ids, urls, item, i, render_content,
                extra_attrs=extra_item_attrs(item) if extra_item_attrs else None
            )
            for i, item in enumerate(queue_items)
        ]
        body = Ul(
            *li_items,
            id=ids.list,
            hx_post=urls.reorder,
            hx_trigger="end",
            hx_include="this",
            hx_target=ids.as_selector(ids.container),
            hx_swap="outerHTML",
            cls=combine_classes("sortable", grow(), overflow.y.auto, list_style.none, m(0), p(0))
        )
    else:
        empty_fn = render_empty or _render_default_empty
        body = Div(empty_fn(), id=ids.empty, cls=combine_classes(grow()))
    
    # Footer (optional)
    footer = render_footer(queue_items) if render_footer else None
    
    return Div(
        header,
        body,
        footer,
        id=ids.container,
        cls=combine_classes(
            panels.minimal_container,
            flex_display,
            flex_direction.col,
            overflow.hidden,
            *container_classes
        )
    )

## Tests

In [ ]:
from fasthtml.common import to_xml

# Test setup
config = SortableQueueConfig(prefix="test")
ids_obj = SortableQueueHtmlIds(prefix="test")
urls = SortableQueueUrls(reorder="/q/reorder", remove="/q/remove", clear="/q/clear")
test_items = [{"id": "a", "name": "Alpha"}, {"id": "b", "name": "Beta"}]

def test_render_content(item, index):
    return Span(item["name"], cls="grow")

# --- render_queue_item ---
li_xml = to_xml(render_queue_item(config, ids_obj, urls, test_items[0], 0, test_render_content))
assert 'id="test-queue-item-a"' in li_xml
assert 'name="item"' in li_xml and 'value="a"' in li_xml  # Hidden input
assert "drag-handle" in li_xml  # Drag handle class
assert "1." in li_xml  # 1-based position
assert "Alpha" in li_xml  # Custom content
assert "/q/remove" in li_xml  # Remove button target
assert 'queue-item' in li_xml  # Item class

# --- render_sortable_queue with items ---
panel_xml = to_xml(render_sortable_queue(config, ids_obj, urls, test_items, test_render_content))
assert 'id="test-queue-container"' in panel_xml
assert 'id="test-queue-list"' in panel_xml
assert 'hx-trigger="end"' in panel_xml
assert 'hx-include="this"' in panel_xml  # HTMX 2.x requirement
assert "Selected" in panel_xml  # Title
assert "/q/reorder" in panel_xml  # Reorder URL
assert "/q/clear" in panel_xml  # Clear button
assert "Alpha" in panel_xml and "Beta" in panel_xml  # Both items rendered
assert "sortable" in panel_xml  # Sortable class on ul

# --- render_sortable_queue empty state ---
empty_xml = to_xml(render_sortable_queue(config, ids_obj, urls, [], test_render_content))
assert 'id="test-queue-container"' in empty_xml
assert 'id="test-queue-empty"' in empty_xml
assert "No items selected" in empty_xml
assert "test-queue-list" not in empty_xml  # No list when empty
assert "/q/clear" not in empty_xml  # No clear button when empty

# --- Custom empty state ---
def custom_empty():
    return P("Nothing here!")

custom_empty_xml = to_xml(render_sortable_queue(config, ids_obj, urls, [], test_render_content, render_empty=custom_empty))
assert "Nothing here!" in custom_empty_xml

# --- Custom footer ---
def test_footer(items_list):
    return Div(f"{len(items_list)} items total", id="footer")

footer_xml = to_xml(render_sortable_queue(config, ids_obj, urls, test_items, test_render_content, render_footer=test_footer))
assert "2 items total" in footer_xml

print("All rendering tests passed")

All rendering tests passed


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()